In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load final feature-selected data
df = pd.read_csv("../data/processed/final_features.csv")

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

X = df.drop(columns=['Converted'])
y = df['Converted']

# Train on FULL dataset now (since this is our final production model)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

final_model = LogisticRegression(max_iter=1000, random_state=42)
final_model.fit(X_scaled, y)

print("✅ Final model trained on full dataset")
print("Features used:", list(X.columns))

✅ Final model trained on full dataset
Features used: ['Total Time Spent on Website', 'Page Views Per Visit', 'TotalVisits', 'What is your current occupation_Working Professional', 'Lead Origin_Lead Add Form', 'What matters most to you in choosing a course_Unknown', 'What is your current occupation_Unknown', 'Lead Source_Reference', 'What is your current occupation_Unemployed', 'Specialization_Unknown', 'Do Not Email', 'Lead Origin_Landing Page Submission', 'Lead Source_Google', 'A free copy of Mastering The Interview', 'City_Thane & Outskirts', 'City_Unknown', 'City_Other Cities', 'Lead Source_Olark Chat', 'Specialization_Human Resource Management', 'Lead Source_Organic Search', 'Lead Source_Welingak Website', 'Specialization_Finance Management', 'Specialization_Marketing Management', 'Country_Unknown', 'City_Other Metro Cities', 'Specialization_Operations Management', 'City_Other Cities of Maharashtra', 'Specialization_Business Administration', 'Specialization_Supply Chain Management'

In [2]:
# Save everything needed for future predictions
joblib.dump(final_model, "../models/lead_scoring_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(list(X.columns), "../models/feature_columns.pkl")

print("✅ Model, scaler, and feature list saved to /models folder")

✅ Model, scaler, and feature list saved to /models folder


In [3]:
def predict_lead_score(new_lead_df, model, scaler, feature_columns):
    """
    Takes a new lead's data (as a DataFrame with the same 30 features)
    and returns conversion probability + prediction.
    """
    # Ensure columns match training data exactly (fill missing with 0)
    new_lead_df = new_lead_df.reindex(columns=feature_columns, fill_value=0)
    
    # Scale
    new_lead_scaled = scaler.transform(new_lead_df)
    
    # Predict
    probability = model.predict_proba(new_lead_scaled)[:, 1]
    prediction = model.predict(new_lead_scaled)
    
    return probability, prediction

print("✅ Prediction function ready")

✅ Prediction function ready


In [4]:
# Test using one row from our dataset (simulating a "new" lead)
sample_lead = X.iloc[[0]]  # double brackets keep it as a DataFrame

prob, pred = predict_lead_score(sample_lead, final_model, scaler, list(X.columns))

print(f"Conversion Probability: {prob[0]*100:.2f}%")
print(f"Prediction: {'Will Convert ✅' if pred[0]==1 else 'Will Not Convert ❌'}")

Conversion Probability: 29.58%
Prediction: Will Not Convert ❌


In [5]:
def generate_lead_scoring_report(new_leads_df, model, scaler, feature_columns):
    """
    Scores multiple leads and returns a ranked report.
    """
    probs, preds = predict_lead_score(new_leads_df, model, scaler, feature_columns)
    
    report = new_leads_df.copy()
    report['Conversion_Probability'] = (probs * 100).round(2)
    report['Prediction'] = np.where(preds == 1, 'Will Convert', 'Will Not Convert')
    
    # Add priority tiers for sales team
    def get_priority(prob):
        if prob >= 70:
            return 'Hot 🔥'
        elif prob >= 40:
            return 'Warm 🌤️'
        else:
            return 'Cold ❄️'
    
    report['Priority'] = report['Conversion_Probability'].apply(get_priority)
    
    # Sort by highest probability first
    report = report.sort_values(by='Conversion_Probability', ascending=False)
    
    return report[['Conversion_Probability', 'Prediction', 'Priority']]

print("✅ Reporting function ready")

✅ Reporting function ready


In [6]:
# Test with 20 random leads from our dataset (simulating new incoming leads)
sample_batch = X.sample(20, random_state=1)

report = generate_lead_scoring_report(sample_batch, final_model, scaler, list(X.columns))
print(report)

      Conversion_Probability        Prediction Priority
2140                   91.51      Will Convert    Hot 🔥
2397                   87.84      Will Convert    Hot 🔥
8100                   85.89      Will Convert    Hot 🔥
7707                   75.56      Will Convert    Hot 🔥
288                    71.41      Will Convert    Hot 🔥
1103                   63.71      Will Convert  Warm 🌤️
1522                   54.70      Will Convert  Warm 🌤️
5163                   50.38      Will Convert  Warm 🌤️
7964                   37.53  Will Not Convert  Cold ❄️
6112                   36.83  Will Not Convert  Cold ❄️
1879                   36.67  Will Not Convert  Cold ❄️
2257                   30.07  Will Not Convert  Cold ❄️
2226                   29.58  Will Not Convert  Cold ❄️
5154                   24.17  Will Not Convert  Cold ❄️
1178                   23.62  Will Not Convert  Cold ❄️
3361                   16.31  Will Not Convert  Cold ❄️
1873                   13.10  Will Not Convert  

In [7]:
report.to_csv("../outputs/lead_scoring_report.csv", index=True)
print("✅ Lead scoring report saved to outputs/lead_scoring_report.csv")

✅ Lead scoring report saved to outputs/lead_scoring_report.csv
